# m4 on Colab: DistilBERT and LoRA runs

[Open in Colab](https://colab.research.google.com/github/jbouniol/nlp-project/blob/main/colab/run_m4.ipynb)

**How to use.** *Runtime → Change runtime type → GPU*, then *Runtime → Run all*. Everything is written to
Google Drive (`MyDrive/ledgar/`). If the session is cut, reconnect and *Run all* again: finished runs are
skipped, an interrupted run resumes from its last epoch.

**Protocol.** Steps 1 and 2 write validation predictions only. The cell after them picks, for each method,
the learning rate with the best validation macro-F1 and saves that choice to `MyDrive/ledgar/choice.json`;
nothing in this notebook reads a test score to make a choice. The test split is then predicted once per chosen
run (step 3), and the later runs use the chosen learning rates. Do not edit hyperparameters in the cells.

| step | runs | free T4 | A100 (Pro) |
|---|---|---|---|
| 1 | full fine-tuning, lr 2e-5, 3e-5, 5e-5 (validation only) | ~25 min each | ~7 min each |
| 2 | LoRA r=8 on q, v, lr 3e-4, 5e-4, 1e-3 (validation only) | ~20 min each | ~5 min each |
| — | automatic choice of the two learning rates on validation | — | — |
| 3 | test predictions of the two chosen runs | ~2 min each | ~1 min each |
| 4 | seeds 1 and 2 of both chosen configurations (with test) | ~1.5 h | ~25 min |
| 5 | LoRA r=16 and LoRA on every linear layer, then their test predictions | ~45 min | ~12 min |
| 6 | optional: `bert-base-uncased` at the chosen full lr | ~50 min | ~15 min |

Total about 5 h on a free T4 (expect two sessions), 1.5 h on an A100. **Afterwards**: copy
`MyDrive/ledgar/results/` into the repository's `results/` (same names) and `MyDrive/ledgar/choice.json`
next to it. Full fine-tuned weights stay on Drive (`MyDrive/ledgar/models/`, 260 MB each).

## Setup

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/ledgar"
RES = f"{DRIVE}/results"
COMMON = f"--out-dir {RES} --checkpoint-dir {DRIVE}/checkpoints --models-dir {DRIVE}/models"

In [ ]:
# The repository at the head of main; Colab's own torch is kept (its CUDA build matches the GPU)
!test -d /content/nlp-project || git clone -q -b main https://github.com/jbouniol/nlp-project.git /content/nlp-project
%cd /content/nlp-project
!git pull -q && git log --oneline -1
!pip install -q transformers==5.19.0 datasets==5.1.0 huggingface-hub==1.33.0 accelerate==1.15.0 peft==0.21.2 scikit-learn==1.9.1
!pip install -q -e . --no-deps

In [ ]:
import json
import os

import torch

print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
assert torch.cuda.is_available(), "Runtime -> Change runtime type -> GPU"


def done(name, final=False):
    # a search run is done when its metrics exist; a final run when its test predictions exist
    return os.path.exists(f"{RES}/{name}/pred_test.npz" if final else f"{RES}/search/{name}/metrics.json")

## Step 1: full fine-tuning, learning rate on validation

In [ ]:
for lr in ("2e-5", "3e-5", "5e-5"):
    name = f"bert_lr{lr}_seed0"
    if not done(name) and not done(name, final=True):
        !python scripts/train.py --model bert --search --lr-full {lr} --seed 0 --name {name} {COMMON}

## Step 2: LoRA r = 8 on `q_lin`, `v_lin`, learning rate on validation

In [ ]:
for lr in ("3e-4", "5e-4", "1e-3"):
    name = f"lora_lr{lr}_seed0"
    if not done(name) and not done(name, final=True):
        !python scripts/train.py --model lora --search --lr-lora {lr} --seed 0 --name {name} {COMMON}

## Choice of the learning rates (validation macro-F1 only)

The rule written before the runs: for each method, the learning rate with the best validation macro-F1. The
choice is saved once to `choice.json` and reused if the notebook is run again.

In [ ]:
choice_path = f"{DRIVE}/choice.json"
if os.path.exists(choice_path):
    choice = json.load(open(choice_path))
else:
    choice = {}
    for method, lrs in (("bert", ("2e-5", "3e-5", "5e-5")), ("lora", ("3e-4", "5e-4", "1e-3"))):
        scores = {}
        for lr in lrs:
            run = f"{method}_lr{lr}_seed0"
            path = f"{RES}/search/{run}/metrics.json"
            if not os.path.exists(path):
                path = f"{RES}/{run}/metrics.json"
            scores[lr] = json.load(open(path))["val"]["macro_f1"]  # fails loudly if a run is missing
        choice[method] = {"lr": max(scores, key=scores.get), "val_macro_f1": scores}
    json.dump(choice, open(choice_path, "w"), indent=2)
LR_FULL, LR_LORA = choice["bert"]["lr"], choice["lora"]["lr"]
print(json.dumps(choice, indent=2))
print("chosen learning rates: full", LR_FULL, "| LoRA", LR_LORA)

## Step 3: test predictions of the chosen runs (once)

In [ ]:
for name, models in ((f"bert_lr{LR_FULL}_seed0", f"--models-dir {DRIVE}/models"), (f"lora_lr{LR_LORA}_seed0", "")):
    if not done(name, final=True):
        !python scripts/predict_test.py --name {name} --out-dir {RES} {models}

## Step 4: seeds 1 and 2 of the chosen configurations (final runs, with test)

In [ ]:
for seed in (1, 2):
    name = f"bert_lr{LR_FULL}_seed{seed}"
    if not done(name, final=True):
        !python scripts/train.py --model bert --lr-full {LR_FULL} --seed {seed} --name {name} {COMMON}
    name = f"lora_lr{LR_LORA}_seed{seed}"
    if not done(name, final=True):
        !python scripts/train.py --model lora --lr-lora {LR_LORA} --seed {seed} --name {name} {COMMON}

## Step 5: LoRA variants at the chosen LoRA learning rate, then their test predictions

Rank 16 with α = 2r (the effective step is unchanged), and rank 8 on every linear layer of the blocks.

In [ ]:
variants = {"lora_r16_seed0": "--lora-r 16 --lora-alpha 32",
            "lora_all_seed0": "--lora-targets q_lin k_lin v_lin out_lin lin1 lin2"}
for name, flags in variants.items():
    if not done(name) and not done(name, final=True):
        !python scripts/train.py --model lora --search --lr-lora {LR_LORA} {flags} --seed 0 --name {name} {COMMON}
    if not done(name, final=True):
        !python scripts/predict_test.py --name {name} --out-dir {RES}

## Step 6 (optional): `bert-base-uncased`, the 110 M-parameter reference, at the chosen full lr

In [ ]:
RUN_BERT_BASE = False  # set to True if there is GPU time left
name = "bertbase_seed0"
if RUN_BERT_BASE and not done(name, final=True):
    if not done(name):
        !python scripts/train.py --model bert --search --model-name bert-base-uncased --lr-full {LR_FULL} --seed 0 --name {name} {COMMON}
    !python scripts/predict_test.py --name {name} --out-dir {RES} --models-dir {DRIVE}/models

## Summary of the runs on Drive (validation and, for final runs, test)

In [ ]:
import glob

for path in sorted(glob.glob(f"{RES}/**/metrics.json", recursive=True)):
    m = json.load(open(path))
    test = m.get("test", {}).get("macro_f1")
    print(f"{path.split('/results/')[1]:<40} val {m['val']['macro_f1']:.4f}" + (f" | test {test:.4f}" if test else ""))